# 01. Подготовка данных

**Цель:** загрузить тензор признаков и target, нормализовать, нарезать на тайлы для обучения.

**Входы:**
- `data/tensor_01deg_v2.npz` (~200 МБ) — тензор X (T=14, H, W, F=20), годы 2010-2023
- `data/y_new_rk_landcover.npz` — target y_new (T, H, W) — TTOP с landcover-varying rk

**Выходы:**
- `data/train_val_pairs.npz` — train/val пары после нарезки на тайлы
- `data/normalization_stats.npz` — статистики для денормализации

In [ ]:
# Environment auto-detection
# Работает на Colab VM, локальном Jupyter, и Colab UI с local runtime
import os
from pathlib import Path

IN_COLAB_VM = (
    'COLAB_RELEASE_TAG' in os.environ or
    'COLAB_GPU' in os.environ
)

if IN_COLAB_VM:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    BASE_DIR = Path('/content/drive/MyDrive/AI4Arctic')
    env_label = 'Colab VM'
else:
    BASE_DIR = Path(os.environ.get(
        'AI4ARCTIC_HOME',
        Path.home() / 'Ai4Arctic'
    ))
    env_label = 'Local runtime'

print(f"Environment: {env_label}")
print(f"BASE_DIR: {BASE_DIR}")
assert BASE_DIR.exists(), f"BASE_DIR не найден: {BASE_DIR}"

import sys
sys.path.insert(0, str(BASE_DIR))

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

# Device: CUDA → MPS (Apple Silicon) → CPU
if torch.cuda.is_available():
    device = 'cuda'
elif torch.backends.mps.is_available():
    device = 'mps'
else:
    device = 'cpu'

print(f"Device: {device}")
print(f"PyTorch: {torch.__version__}")

# Пути
DATA_DIR = BASE_DIR / 'data'
MODELS_DIR = BASE_DIR / 'models'
RESULTS_DIR = BASE_DIR / 'results'
FIGURES_DIR = RESULTS_DIR / 'figures'
METRICS_DIR = RESULTS_DIR / 'metrics'

for d in [MODELS_DIR, FIGURES_DIR, METRICS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

## 1. Загружаем тензор и target

In [ ]:
TENSOR_PATH = DATA_DIR / 'tensor_01deg_v2.npz'
TARGET_PATH = DATA_DIR / 'y_new_rk_landcover.npz'

print(f"Загружаем {TENSOR_PATH}")
data = np.load(TENSOR_PATH)
X_full = data['X'].astype(np.float32)
print(f"  X: {X_full.shape}")  # ожидаем (14, 231, 1501, 20)

print(f"\nЗагружаем {TARGET_PATH}")
y_new_data = np.load(TARGET_PATH)
y_full = y_new_data['y_new'].astype(np.float32)
rk_map = y_new_data['rk_map']
print(f"  y_new: {y_full.shape}")
print(f"  rk_map: {rk_map.shape}, min={rk_map.min():.3f}, max={rk_map.max():.3f}")

## 2. Нормализация признаков и target

In [ ]:
from src.data import normalize_features, normalize_target

# Per-channel mean/std считаем по первым 13 годам (train), валидация на 13-м (2023)
X_norm, x_mean, x_std = normalize_features(X_full, train_years_end=13, n_features=20)
print(f"X_norm: {X_norm.shape}, dtype={X_norm.dtype}")
print(f"x_mean[:5]: {x_mean[:5]}")
print(f"x_std[:5]: {x_std[:5]}")

y_norm, y_nan_mask, y_mean, y_std = normalize_target(y_full, train_years_end=13)
print(f"\ny_norm: {y_norm.shape}")
print(f"y_mean={y_mean:.4f}, y_std={y_std:.4f}")
print(f"NaN пикселей в y: {y_nan_mask.sum():,} ({100*y_nan_mask.mean():.1f}%)")

## 3. Нарезка на тайлы + train/val split

In [ ]:
from src.data import make_pairs

# Train: target_years = 4..12 (всего 9 пар, с input_len=4)
# Val:   target_year = 13 (2023)
train_targets = list(range(4, 13))
val_targets = [13]

print(f"Train targets (по индексу года в тензоре): {train_targets}")
print(f"Val targets: {val_targets}\n")

Xtr, ytr, mtr = make_pairs(X_norm, y_norm, y_nan_mask, train_targets)
Xv, yv, mv = make_pairs(X_norm, y_norm, y_nan_mask, val_targets)

print(f"Train: {Xtr.shape[0]} тайлов, X={Xtr.shape}, y={ytr.shape}, m={mtr.shape}")
print(f"Val:   {Xv.shape[0]} тайлов")

## 4. Сохраняем для последующего обучения

In [ ]:
PAIRS_PATH = DATA_DIR / 'train_val_pairs.npz'
NORM_PATH = DATA_DIR / 'normalization_stats.npz'

np.savez_compressed(
    PAIRS_PATH,
    Xtr=Xtr, ytr=ytr, mtr=mtr,
    Xv=Xv, yv=yv, mv=mv,
)
print(f"Сохранено: {PAIRS_PATH} ({PAIRS_PATH.stat().st_size/1e6:.1f} МБ)")

np.savez(
    NORM_PATH,
    x_mean=x_mean, x_std=x_std,
    y_mean=y_mean, y_std=y_std,
    rk_map=rk_map,
)
print(f"Сохранено: {NORM_PATH}")

## 5. Визуальный sanity-check

Что-то странное → не идём в обучение.

In [ ]:
# Распределение значений 1 случайного тайла
sample_idx = 0
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

# Каналы X
for c, (ax, name) in enumerate(zip(axes[0], ['NDVI', 'NDWI', 'NDMI (norm.)'])):
    im = ax.imshow(Xtr[sample_idx, -1, :, :, c], cmap='RdYlGn')
    ax.set_title(f'X channel {c}: {name}')
    plt.colorbar(im, ax=ax, fraction=0.046)

# Target и mask
axes[1, 0].imshow(ytr[sample_idx], cmap='RdBu_r', vmin=-3, vmax=3)
axes[1, 0].set_title('y (normalized)')

axes[1, 1].imshow(mtr[sample_idx], cmap='gray')
axes[1, 1].set_title(f'mask (valid fraction: {mtr[sample_idx].mean():.2f})')

axes[1, 2].hist(ytr[mtr > 0.5], bins=50)
axes[1, 2].set_title('y values distribution')
axes[1, 2].set_xlabel('normalized y')

plt.tight_layout()
plt.show()